# Day 13 — Text Embeddings and Semantic Search
## 30 Days of AI: From NLP to LLMs

---

On Day 12 you learned to talk to LLMs through prompts.
But LLMs have a fundamental limitation: they only know what
is inside their context window. If the answer is in a 500-page
PDF you have not sent, the model cannot help you.

The solution is retrieval — finding the right piece of information
and injecting it into the prompt. But how do you find the right
piece? Keyword search fails when the user asks 'what causes
headaches' and the document says 'cephalgia triggers'. The words
are different but the meaning is identical.

Text embeddings solve this. An embedding is a dense vector that
represents the MEANING of text, not its exact words. Similar
meanings produce similar vectors — measurable with cosine similarity.
This is the retrieval engine that powers RAG on Days 14 and 15.

---

### What You Will Learn Today

- What embeddings are and why dense vectors encode meaning
- How to generate embeddings with sentence-transformers
- Cosine similarity — the geometry of meaning
- Building a semantic search engine from scratch
- FAISS — fast approximate nearest-neighbor search at scale
- Embedding models comparison — size vs quality trade-offs
- Semantic vs keyword search — when each wins

### Goal by End of Day

Build a working semantic search engine over a document corpus.
Query it with natural language and retrieve the most relevant
passages by meaning, not keywords. Understand why FAISS is
necessary at scale and how it trades accuracy for speed.

In [ ]:
## Run once
## !pip install sentence-transformers faiss-cpu numpy matplotlib scikit-learn -q

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

import torch
print('PyTorch      :', torch.__version__)

from sentence_transformers import SentenceTransformer
print('SentenceTransformers ready.')

try:
    import faiss
    print('FAISS        : available')
    FAISS_AVAILABLE = True
except ImportError:
    print('FAISS        : not installed — run: pip install faiss-cpu')
    FAISS_AVAILABLE = False

---

## Part 1 — What Are Embeddings?

An embedding is a fixed-size dense vector that represents the
semantic content of a piece of text. The key insight:

```
Text  →  [Embedding Model]  →  Vector of floats

'A dog ran across the yard'  →  [0.21, -0.54, 0.87, ..., 0.13]  (384 dims)
'A puppy sprinted through the garden'  →  [0.22, -0.51, 0.85, ..., 0.14]
'The stock market crashed yesterday'   →  [-0.43, 0.91, -0.12, ..., 0.78]
```

Sentences with similar meaning produce vectors that are close in
the embedding space. Unrelated sentences produce vectors that are far.

### From word2vec to Sentence Embeddings

```
word2vec (2013)     :  one vector per WORD
                       'bank' has ONE vector regardless of context
                       ignores 'river bank' vs 'savings bank'

BERT (2018)         :  contextual word embeddings
                       'bank' gets different vectors in different sentences
                       but outputs one vector per TOKEN, not per sentence

Sentence-BERT (2019):  fine-tuned BERT using siamese networks
                       trained on Natural Language Inference pairs
                       outputs ONE vector per sentence — perfect for retrieval

Modern embedding models (2022-2024):
                       OpenAI text-embedding-3, Cohere Embed, E5, BGE
                       trained on massive datasets with contrastive learning
                       state-of-art performance on retrieval benchmarks
```

### The Geometry of Meaning

```
Embeddings live in a high-dimensional space (128 to 4096 dims).
Direction encodes meaning. Distance encodes similarity.

Classic word2vec arithmetic:
  king - man + woman  ≈  queen
  Paris - France + Italy  ≈  Rome

This generalizes to sentences:
  'How to bake bread' is close to 'bread baking instructions'
  Both are far from 'quarterly earnings report'
```

In [ ]:
# ---------------------------------------------------------------
# Load a sentence embedding model
# all-MiniLM-L6-v2 : 22M params, 384 dims, very fast, great quality
# Best speed/quality ratio for learning and prototyping
# ---------------------------------------------------------------

MODEL_NAME = 'all-MiniLM-L6-v2'
model = SentenceTransformer(MODEL_NAME)

print(f'Model        : {MODEL_NAME}')
print(f'Dimensions   : {model.get_sentence_embedding_dimension()}')
print(f'Max tokens   : {model.max_seq_length}')
print()

# Embed a single sentence
sentence = 'The Transformer architecture revolutionized natural language processing.'
embedding = model.encode(sentence)

print(f'Input text   : "{sentence}"')
print(f'Output shape : {embedding.shape}')
print(f'Output type  : {embedding.dtype}')
print(f'First 8 dims : {embedding[:8].round(4)}')
print(f'Vector norm  : {np.linalg.norm(embedding):.4f}')
print()
print('The vector norm is close to 1.0 — this model returns unit vectors.')
print('For unit vectors: cosine_similarity = dot_product (faster to compute)')

---

## Part 2 — Cosine Similarity

Cosine similarity measures the angle between two vectors.
It is the standard metric for comparing embeddings.

```
cosine_similarity(A, B)  =  (A · B) / (|A| × |B|)

Range  :  -1.0  to  +1.0

  +1.0  →  identical direction  →  same meaning
   0.0  →  orthogonal          →  unrelated meaning
  -1.0  →  opposite direction  →  opposite meaning

In practice for text embeddings:
  > 0.9   very similar — near paraphrases
  0.7-0.9 related topic, similar theme
  0.5-0.7 loosely related
  < 0.5   unrelated

Why cosine and not Euclidean distance?
  Euclidean distance is affected by vector magnitude (length).
  A long document embedding would be 'far' from a short one
  even if they discuss the same topic.
  Cosine only cares about direction — not length.
  For unit-normalized embeddings: cosine similarity = dot product.
```

In [ ]:
# ---------------------------------------------------------------
# Compute and visualize cosine similarity between sentences
# ---------------------------------------------------------------

sentences = [
    # Group 1: Machine learning
    'Deep learning models require large amounts of training data.',
    'Neural networks need many examples to learn effectively.',
    'Training AI requires huge datasets.',
    # Group 2: Finance
    'The stock market fell sharply after the interest rate announcement.',
    'Equity prices dropped following the central bank decision.',
    'Investors sold off shares after the Fed raised rates.',
    # Group 3: Cooking
    'Preheat the oven to 200 degrees before baking the bread.',
    'Bake at 200 Celsius until the crust is golden brown.',
    'The recipe requires the oven to reach full temperature first.',
]

# Embed all sentences at once (batched — much faster)
embeddings = model.encode(sentences, normalize_embeddings=True)

# Compute full similarity matrix
sim_matrix = cosine_similarity(embeddings)

# Plot heatmap
short_labels = [
    'ML: needs data', 'ML: needs examples', 'ML: requires datasets',
    'Finance: market fell', 'Finance: prices dropped', 'Finance: investors sold',
    'Cook: preheat oven', 'Cook: bake 200C', 'Cook: full temp first',
]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)

for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}',
                ha='center', va='center', fontsize=7,
                color='black' if sim_matrix[i,j] > 0.3 else 'gray')

ax.set_title('Cosine Similarity Matrix\n'
             'Same-topic sentences (3x3 diagonal blocks) score high.\n'
             'Cross-topic sentences score low.', fontsize=10)
plt.tight_layout()
plt.show()

print('Key observations:')
print(f'  ML sentences similarity   : {sim_matrix[0,1]:.3f}, {sim_matrix[0,2]:.3f}')
print(f'  Finance sentences sim     : {sim_matrix[3,4]:.3f}, {sim_matrix[3,5]:.3f}')
print(f'  ML vs Finance cross-topic : {sim_matrix[0,3]:.3f}')
print(f'  ML vs Cooking cross-topic : {sim_matrix[0,6]:.3f}')

In [ ]:
# ---------------------------------------------------------------
# Visualize embeddings in 2D using PCA
# PCA reduces 384 dims to 2 for visualization
# ---------------------------------------------------------------

pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embeddings)

colors  = ['steelblue'] * 3 + ['tomato'] * 3 + ['forestgreen'] * 3
markers = ['o'] * 3 + ['s'] * 3 + ['^'] * 3
group_labels = ['Machine Learning'] * 3 + ['Finance'] * 3 + ['Cooking'] * 3

fig, ax = plt.subplots(figsize=(9, 6))

seen_labels = set()
for i, (x, y) in enumerate(coords_2d):
    label = group_labels[i] if group_labels[i] not in seen_labels else None
    seen_labels.add(group_labels[i])
    ax.scatter(x, y, c=colors[i], marker=markers[i], s=120,
               label=label, zorder=3)
    ax.annotate(
        sentences[i][:35] + '...',
        (x, y), textcoords='offset points',
        xytext=(8, 4), fontsize=7, color=colors[i]
    )

ax.legend(fontsize=9)
ax.set_title(
    'Sentence Embeddings Projected to 2D (PCA)\n'
    'Same-topic sentences cluster together in embedding space.',
    fontsize=10
)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Each cluster in the 2D plot corresponds to a topic group.')
print('This is the core property that makes semantic search work.')

---

## Part 3 — Building a Semantic Search Engine

A semantic search engine has three components:

```
OFFLINE  (done once, can take minutes or hours)
─────────────────────────────────────────────────
1. Chunk documents into passages
   → Split text into ~200-500 token chunks
   → Overlap chunks so context is not lost at boundaries

2. Embed each chunk
   → Run all chunks through the embedding model
   → Store vectors in a vector index

ONLINE  (done per query, must be fast — <100ms)
─────────────────────────────────────────────────
3. Embed the query
   → Same embedding model, same normalization

4. Find nearest neighbors
   → Compare query vector to all document vectors
   → Return top-k most similar chunks

5. Return results (for search) or pass to LLM (for RAG)
```

### Why Chunking Matters

```
Too large chunks  (1000+ tokens):
  → One vector represents many topics
  → Similarity gets diluted — irrelevant content drags down the score
  → Hard for LLM to find the answer in the large retrieved block

Too small chunks  (<50 tokens):
  → Not enough context for the embedding to capture meaning
  → Many chunks needed to cover the same content
  → Retrieved snippets may be incomplete sentences

Sweet spot: 200-400 tokens per chunk
  → Enough context for semantic richness
  → Small enough to be specific
  → 50-token overlap between chunks preserves cross-boundary context
```

In [ ]:
# ---------------------------------------------------------------
# A realistic document corpus — 20 passages across 4 topics
# ---------------------------------------------------------------

CORPUS = [
    # --- Machine Learning ---
    'Gradient descent is an optimization algorithm that iteratively\
 adjusts model parameters to minimize a loss function by moving\
 in the direction of the steepest descent.',

    'Overfitting occurs when a model learns the training data too well,\
 capturing noise and outliers. It generalizes poorly to unseen data.\
 Regularization techniques like dropout and L2 penalty help prevent it.',

    'Batch normalization stabilizes neural network training by normalizing\
 the inputs to each layer. It reduces internal covariate shift and\
 allows higher learning rates.',

    'The learning rate controls how large a step gradient descent takes\
 at each iteration. Too high and training diverges. Too low and\
 training is slow. Learning rate schedules adapt it during training.',

    'Cross-entropy loss is the standard loss function for classification.\
 It measures the difference between the predicted probability distribution\
 and the true one-hot label distribution.',

    # --- Natural Language Processing ---
    'Tokenization splits raw text into tokens — the basic units a\
 language model processes. Subword tokenization like BPE handles\
 rare words by splitting them into known subpieces.',

    'Named entity recognition identifies and classifies named entities\
 in text such as people, organizations, and locations. BERT-based\
 models achieve state-of-the-art NER results.',

    'Text summarization produces a shorter version of a document while\
 preserving key information. Extractive methods select existing sentences.\
 Abstractive methods generate new text.',

    'Sentiment analysis classifies the emotional tone of text as\
 positive, negative, or neutral. It is widely used in customer\
 feedback analysis and social media monitoring.',

    'Machine translation uses sequence-to-sequence models to convert\
 text from one language to another. Modern systems like Google Translate\
 are based on Transformer encoder-decoder architectures.',

    # --- Python Programming ---
    'List comprehensions in Python provide a concise way to create lists.\
 The syntax [expression for item in iterable if condition] is faster\
 than equivalent for-loop code for most operations.',

    'Python decorators are functions that modify the behavior of other\
 functions. The @decorator syntax applies them. Common uses include\
 logging, authentication, and caching.',

    'Context managers in Python using the with statement ensure proper\
 resource cleanup. They implement __enter__ and __exit__ methods.\
 The most common use is opening and closing files safely.',

    'Python generators use the yield keyword to return values lazily,\
 one at a time. They are memory-efficient for large sequences because\
 they do not store all values in memory at once.',

    'Virtual environments in Python isolate project dependencies.\
 The venv module creates isolated Python installations. This prevents\
 version conflicts between different projects.',

    # --- Data Science ---
    'The train-test split divides a dataset into a training set used\
 to fit the model and a test set used to evaluate final performance.\
 A common ratio is 80% training and 20% test.',

    'Feature scaling normalizes numerical features to a common range.\
 StandardScaler removes the mean and scales to unit variance.\
 MinMaxScaler maps values to [0, 1].',

    'Confusion matrices show the counts of true positives, true negatives,\
 false positives, and false negatives. They reveal which classes the\
 model confuses with each other.',

    'K-fold cross-validation splits data into k equal folds and trains\
 k models, each using a different fold as the validation set.\
 The average score is a robust estimate of generalization.',

    'Principal Component Analysis (PCA) reduces dimensionality by\
 projecting data onto the directions of maximum variance.\
 It is used for visualization, noise reduction, and compression.',
]

CORPUS_TOPICS = (
    ['Machine Learning'] * 5 +
    ['NLP'] * 5 +
    ['Python'] * 5 +
    ['Data Science'] * 5
)

print(f'Corpus size : {len(CORPUS)} passages')
print(f'Topics      : {set(CORPUS_TOPICS)}')
print()
print('Sample passage:')
print(' ', CORPUS[0])

In [ ]:
# ---------------------------------------------------------------
# Step 1: Embed the entire corpus (offline step)
# ---------------------------------------------------------------

import time

start = time.time()
corpus_embeddings = model.encode(
    CORPUS,
    normalize_embeddings = True,   # unit vectors → dot product = cosine sim
    show_progress_bar    = True,
    batch_size           = 32,
)
elapsed = time.time() - start

print()
print('Corpus Embeddings')
print('=' * 50)
print(f'Shape         : {corpus_embeddings.shape}')
print(f'  → {len(CORPUS)} passages × {corpus_embeddings.shape[1]} dims')
print(f'Encoding time : {elapsed:.2f}s  ({elapsed/len(CORPUS)*1000:.1f}ms per passage)')
print(f'Memory usage  : {corpus_embeddings.nbytes / 1024:.1f} KB')
print()
print('In production, these embeddings are stored in a vector database')
print('(FAISS, Pinecone, Weaviate, Chroma) and loaded on startup.')

In [ ]:
# ---------------------------------------------------------------
# Step 2: Semantic search function (numpy version)
# For small corpora (<10K passages) this is fast enough
# ---------------------------------------------------------------

def semantic_search(query, corpus, corpus_embeddings, model, top_k=3):
    """
    Find the top_k most semantically similar passages to the query.

    Steps:
      1. Embed the query
      2. Compute dot product with all corpus embeddings (= cosine sim for unit vectors)
      3. Return top-k highest scoring passages
    """
    # Embed query (returns shape (dim,))
    query_embedding = model.encode(query, normalize_embeddings=True)

    # Dot product: query (dim,) @ corpus (N, dim).T = (N,)
    scores = corpus_embeddings @ query_embedding

    # Get top-k indices
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'rank'    : len(results) + 1,
            'score'   : float(scores[idx]),
            'passage' : corpus[idx],
            'topic'   : CORPUS_TOPICS[idx],
            'index'   : int(idx),
        })
    return results


# Test with several queries
queries = [
    'What prevents a neural network from memorizing training examples?',
    'How do I handle file operations safely in Python?',
    'How do I evaluate the performance of a classifier?',
    'What breaks down long words into pieces for language models?',
]

print('Semantic Search Results')
print('=' * 70)

for query in queries:
    results = semantic_search(query, CORPUS, corpus_embeddings, model, top_k=3)
    print(f'\nQuery  : "{query}"')
    print('-' * 70)
    for r in results:
        print(f'  Rank {r["rank"]} | Score: {r["score"]:.4f} | Topic: {r["topic"]}')
        print(f'  "{r["passage"][:90]}..."')

---

## Part 4 — FAISS: Scaling to Millions of Vectors

The numpy approach works for small corpora. But brute-force
dot product over 1 million 768-dim vectors takes ~2 seconds.
At 10 million vectors — 20 seconds. Not acceptable for production.

FAISS (Facebook AI Similarity Search) solves this with
approximate nearest neighbor (ANN) algorithms.

```
Exact search: check every vector — O(N × d)
  → 1M vectors, 384 dims → ~1.5B operations per query
  → Slow at scale, but 100% accurate

FAISS ANN: partition space, search only nearby regions — O(log N)
  → 1M vectors, same query → ~5ms
  → 95-99% accurate (configurable)

FAISS Index Types:
─────────────────────────────────────────────────────────────
IndexFlatL2          Exact L2 distance. Brute force. Best for <100K vectors.
IndexFlatIP          Exact inner product (= cosine for unit vectors).
IndexIVFFlat         Inverted file index. Clusters vectors, searches clusters.
                     nlist=100 clusters, nprobe=10 clusters searched.
                     Good for 100K-10M vectors.
IndexHNSWFlat        Hierarchical navigable small world graph.
                     Fast search, good recall, high memory.
IndexIVFPQ           Adds product quantization compression.
                     10-40x memory reduction vs flat.
                     Best for 10M+ vectors on constrained hardware.
```

In [ ]:
# ---------------------------------------------------------------
# Build a FAISS index and search it
# ---------------------------------------------------------------

if FAISS_AVAILABLE:
    import faiss

    dim = corpus_embeddings.shape[1]   # 384

    # IndexFlatIP = exact inner product search
    # For unit-normalized vectors this equals cosine similarity
    index = faiss.IndexFlatIP(dim)

    # FAISS requires float32
    vectors = corpus_embeddings.astype(np.float32)

    # Add all corpus vectors to the index
    index.add(vectors)

    print('FAISS Index')
    print('=' * 50)
    print(f'Index type      : IndexFlatIP')
    print(f'Dimensions      : {dim}')
    print(f'Vectors indexed : {index.ntotal}')
    print()

    def faiss_search(query, index, corpus, model, top_k=3):
        """Search FAISS index for nearest neighbors."""
        q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)

        # index.search returns (distances, indices) — shape (1, top_k) each
        scores, indices = index.search(q_emb, top_k)

        results = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
            results.append({
                'rank'    : rank + 1,
                'score'   : float(score),
                'passage' : corpus[idx],
                'topic'   : CORPUS_TOPICS[idx],
            })
        return results

    # Benchmark: FAISS vs numpy
    query = 'How do I stop my model from overfitting?'

    # Numpy timing
    t0 = time.time()
    for _ in range(100):
        semantic_search(query, CORPUS, corpus_embeddings, model, top_k=3)
    numpy_ms = (time.time() - t0) / 100 * 1000

    # FAISS timing
    q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
    t0 = time.time()
    for _ in range(100):
        index.search(q_emb, 3)
    faiss_ms = (time.time() - t0) / 100 * 1000

    print(f'Search speed comparison (100 queries, {len(CORPUS)} passages):')
    print(f'  Numpy brute-force  : {numpy_ms:.2f} ms/query')
    print(f'  FAISS IndexFlatIP  : {faiss_ms:.2f} ms/query')
    print()
    print('FAISS advantage grows enormously with corpus size:')
    print('  10K docs   → ~3x faster')
    print('  1M docs    → ~100x faster  (IVF index)')
    print('  100M docs  → only FAISS is practical')

else:
    print('FAISS not available — install with: pip install faiss-cpu')
    print()
    print('For this corpus size (20 passages), numpy works perfectly.')
    print('FAISS becomes necessary above ~10,000 passages.')

In [ ]:
# ---------------------------------------------------------------
# Save and load a FAISS index
# In production: build once offline, load at server startup
# ---------------------------------------------------------------

if FAISS_AVAILABLE:
    import pickle

    # Save
    faiss.write_index(index, '/tmp/corpus.index')

    # Also save the corpus text (FAISS only stores vectors, not text)
    with open('/tmp/corpus_text.pkl', 'wb') as f:
        pickle.dump({'corpus': CORPUS, 'topics': CORPUS_TOPICS}, f)

    print('Saved:')
    print('  /tmp/corpus.index      — FAISS vector index')
    print('  /tmp/corpus_text.pkl   — original passage text')
    print()

    # Load
    loaded_index = faiss.read_index('/tmp/corpus.index')
    with open('/tmp/corpus_text.pkl', 'rb') as f:
        data = pickle.load(f)

    print(f'Loaded index with {loaded_index.ntotal} vectors')
    print('Index is identical to original:', loaded_index.ntotal == index.ntotal)
else:
    print('Skipped — FAISS not installed.')

---

## Part 5 — Semantic Search vs Keyword Search

```
Keyword Search (BM25, TF-IDF)
────────────────────────────────────────────────────────────
How it works  : count word overlaps between query and document
Wins when     : query uses same words as document
               'python list comprehension syntax'
               → finds docs containing those exact terms
Fails when    : query uses different vocabulary than document
               'fast way to create lists in python'
               → misses docs that say 'list comprehensions'
Pros          : fast, no embedding model needed, exact match reliable
Cons          : vocabulary mismatch, no semantic understanding

Semantic Search (Embeddings)
────────────────────────────────────────────────────────────
How it works  : embed query and documents, find nearest vectors
Wins when     : query and document use different but related words
               'what prevents overfitting' → finds 'regularization'
Fails when    : exact terms matter (product codes, names, IDs)
               'SKU-4821' → embedding gives no advantage
Pros          : handles paraphrase, synonym, multilingual
Cons          : slower indexing, embedding model required

Hybrid Search (Best of Both)
────────────────────────────────────────────────────────────
Combine BM25 score + cosine similarity score with a weight α:
  hybrid_score = α × semantic_score + (1-α) × keyword_score

Used by: Elasticsearch, Weaviate, Pinecone, most prod systems
α = 0.5 is a good starting point, tune on your data
```

In [ ]:
# ---------------------------------------------------------------
# Side-by-side: semantic search vs keyword search (TF-IDF)
# ---------------------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cos

# Build TF-IDF index
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(CORPUS)


def keyword_search(query, tfidf, tfidf_matrix, corpus, top_k=3):
    """BM25-like keyword search using TF-IDF cosine similarity."""
    q_vec   = tfidf.transform([query])
    scores  = sklearn_cos(q_vec, tfidf_matrix)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [
        {'rank': i+1, 'score': float(scores[idx]),
         'passage': corpus[idx], 'topic': CORPUS_TOPICS[idx]}
        for i, idx in enumerate(top_idx)
    ]


# Queries designed to expose the difference
test_cases = [
    {
        'query'  : 'What stops a model from memorizing the training set?',
        'note'   : 'Semantic wins: "memorizing" ≠ "overfitting" in keywords'
    },
    {
        'query'  : 'How does gradient descent work?',
        'note'   : 'Keyword wins: exact match on "gradient descent"'
    },
    {
        'query'  : 'How can I split text into subword units?',
        'note'   : 'Semantic wins: "subword" is in corpus but query says "split text"'
    },
]

print('Semantic vs Keyword Search Comparison')
print('=' * 70)

for case in test_cases:
    q = case['query']
    print(f'\nQuery  : "{q}"')
    print(f'Note   : {case["note"]}')
    print('-' * 70)

    sem_results = semantic_search(q, CORPUS, corpus_embeddings, model, top_k=1)
    kw_results  = keyword_search(q, tfidf, tfidf_matrix, CORPUS, top_k=1)

    print(f'Semantic top-1 (score {sem_results[0]["score"]:.3f}):')
    print(f'  [{sem_results[0]["topic"]}] {sem_results[0]["passage"][:90]}...')

    print(f'Keyword  top-1 (score {kw_results[0]["score"]:.3f}):')
    print(f'  [{kw_results[0]["topic"]}] {kw_results[0]["passage"][:90]}...')

---

## Part 6 — Embedding Models Comparison

```
Model                          Dims   Size    Speed   MTEB Score  Best For
─────────────────────────────────────────────────────────────────────────────
all-MiniLM-L6-v2               384    22M     ★★★★★   59.9        Prototyping, CPU
all-mpnet-base-v2              768    110M    ★★★★    57.8        Better quality, GPU
BAAI/bge-small-en-v1.5         384    33M     ★★★★★   62.2        Best small model
BAAI/bge-large-en-v1.5         1024   335M    ★★★     63.6        High quality
intfloat/e5-large-v2            1024   335M    ★★★     62.5        Strong retrieval
openai/text-embedding-3-small  1536   API     ★★★★    62.3        API convenience
openai/text-embedding-3-large  3072   API     ★★★     64.6        Best via API
cohere/embed-english-v3.0      1024   API     ★★★★    64.5        Production quality

MTEB = Massive Text Embedding Benchmark (56 tasks, standard evaluation)
Scores approximate — check https://huggingface.co/spaces/mteb/leaderboard

Recommendations:
  Learning / prototyping  :  all-MiniLM-L6-v2   (fast, free, good enough)
  Production (local)      :  BAAI/bge-large-en   (best open model)
  Production (API)        :  openai text-embedding-3-large
  Multilingual            :  paraphrase-multilingual-MiniLM-L12-v2
```

In [ ]:
# ---------------------------------------------------------------
# Full semantic search pipeline — production-ready class
# This is the component you will plug into RAG on Day 14
# ---------------------------------------------------------------

class SemanticSearchEngine:
    """
    A complete semantic search engine.
    Supports add, search, save, and load.
    """

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model    = SentenceTransformer(model_name)
        self.dim      = self.model.get_sentence_embedding_dimension()
        self.passages = []
        self.metadata = []

        if FAISS_AVAILABLE:
            self.index = faiss.IndexFlatIP(self.dim)
        else:
            self.embeddings = None   # numpy fallback

    def add(self, passages, metadata=None):
        """Embed and index a list of passages."""
        if metadata is None:
            metadata = [{}] * len(passages)

        embeddings = self.model.encode(
            passages,
            normalize_embeddings = True,
            show_progress_bar    = False,
        ).astype(np.float32)

        self.passages.extend(passages)
        self.metadata.extend(metadata)

        if FAISS_AVAILABLE:
            self.index.add(embeddings)
        else:
            if self.embeddings is None:
                self.embeddings = embeddings
            else:
                self.embeddings = np.vstack([self.embeddings, embeddings])

    def search(self, query, top_k=5):
        """Search for the top_k most similar passages."""
        q_emb = self.model.encode(
            [query], normalize_embeddings=True
        ).astype(np.float32)

        if FAISS_AVAILABLE:
            scores, indices = self.index.search(q_emb, top_k)
            scores, indices = scores[0], indices[0]
        else:
            scores  = self.embeddings @ q_emb[0]
            indices = np.argsort(scores)[::-1][:top_k]
            scores  = scores[indices]

        return [
            {
                'rank'     : i + 1,
                'score'    : float(scores[i]),
                'passage'  : self.passages[indices[i]],
                'metadata' : self.metadata[indices[i]],
            }
            for i in range(len(indices))
        ]

    def __len__(self):
        return len(self.passages)


# Build and test
engine = SemanticSearchEngine()
engine.add(
    CORPUS,
    metadata=[{'topic': t, 'id': i} for i, t in enumerate(CORPUS_TOPICS)]
)

print(f'Engine loaded with {len(engine)} passages.')
print()

test_query = 'How do I evaluate whether my ML model is working well?'
results    = engine.search(test_query, top_k=3)

print(f'Query: "{test_query}"')
print('=' * 65)
for r in results:
    print(f'Rank {r["rank"]} | Score: {r["score"]:.4f} | Topic: {r["metadata"]["topic"]}')
    print(f'  {r["passage"][:100]}...')
    print()

---

## Day 13 Summary

```
What you built today:

1.  Embeddings          →  text → dense vector via SentenceTransformer
2.  Cosine similarity   →  measuring meaning distance, heatmap visualization
3.  PCA projection      →  seeing clusters in 2D from 384-dim space
4.  Semantic search     →  query embedding → dot product → top-k passages
5.  FAISS index         →  IndexFlatIP, add, search, save, load
6.  Keyword vs semantic →  when each wins, hybrid approach
7.  SemanticSearchEngine→  production-ready class with metadata support

```

### Self-Check Questions

Answer these before Day 14:

1. Why do we normalize embedding vectors before computing similarity?
2. A user asks 'how do I fix my code?'. The corpus has 'debugging
   techniques for Python'. Will semantic search find it? Why?
3. What is the trade-off between IndexFlatIP and IndexIVFFlat in FAISS?
4. Why is chunk size important in RAG? What happens at both extremes?
5. Your embedding model has max_seq_length=256 tokens. A passage is
   400 tokens. What happens to the last 144 tokens?